# Lección 4.2 — Métricas de clasificación: más allá de la exactitud

> **Módulo 4 · Evaluación: ¿mi modelo es bueno?** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.7 (datos desbalanceados), Lección 4.1 (métricas de regresión)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Construir y leer una **matriz de confusión** (verdaderos/falsos positivos y negativos).
- Explicar por qué la **exactitud** engaña cuando las clases están desbalanceadas.
- Calcular e interpretar **precisión**, **recall** (sensibilidad), **especificidad** y **F1**.
- Entender por qué precisión y recall **compiten** entre sí.
- Leer un `classification_report`, también con más de dos clases.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score, classification_report)

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: la alarma de humo

Una alarma de humo es un **clasificador**: para cada momento decide "🔥 hay fuego" o "✅ no hay fuego". Puede acertar de dos formas y **equivocarse de dos formas muy distintas**:

| | 🔥 Sí había fuego | ✅ No había fuego |
|---|---|---|
| **🚨 La alarma sonó** | ✅ **Verdadero positivo (VP)**: te salvó | ❌ **Falso positivo (FP)**: sonó por el pan tostado |
| **🔇 La alarma calló** | ❌ **Falso negativo (FN)**: ¡la casa se quema! | ✅ **Verdadero negativo (VN)**: noche tranquila |

Truco para los nombres:
- **Positivo / negativo** = lo que **dijo** el modelo (sonó / calló).
- **Verdadero / falso** = si **acertó** o no.

Fíjate en que los dos errores **no cuestan lo mismo**: un falso positivo es una molestia; un falso negativo puede ser una tragedia. Toda esta lección gira alrededor de esa idea.

### ✋ Ejercicio 1 — Ponle nombre al resultado
En cada caso, lo "positivo" es lo que el sistema intenta detectar. ¿Es VP, FP, FN o VN?

1. Una prueba de embarazo dice "embarazada" y la persona **no** lo está.
2. El filtro de spam deja pasar a tu bandeja un correo de spam.
3. Un detector de metales en el aeropuerto suena y la persona llevaba un cinturón metálico.
4. Un control de calidad dice "pieza buena" y la pieza **es** buena (lo positivo = "pieza defectuosa").

✍️ *Mis respuestas:* 1 → __ · 2 → __ · 3 → __ · 4 → __

<details><summary><b>👀 Ver respuesta</b></summary>

1. **FP**: dijo positivo y se equivocó.
2. **FN**: lo positivo es "spam"; el filtro dijo "no es spam" (negativo) y se equivocó.
3. **VP**: lo positivo es "lleva metal"; sonó y era cierto. (Que el metal sea inofensivo es otro tema: el detector solo busca metal.)
4. **VN**: dijo "no defectuosa" (negativo) y acertó.

> 📝 Lo más importante es decidir **qué clase es la "positiva"**: casi siempre, la que quieres **detectar** (la enfermedad, el spam, la pieza defectuosa). No significa "buena noticia".
</details>

---
## 2. 📖 La matriz de confusión y la trampa de la exactitud

Una fábrica produce piezas metálicas. Algunas salen **defectuosas** y hay que detectarlas antes de enviarlas al cliente. Tenemos 4,000 piezas con tres mediciones de la máquina.

In [ ]:
# ▶️ Historial ficticio de piezas
rng = np.random.default_rng(3)
n = 4_000
piezas = pd.DataFrame({
    "temperatura": rng.normal(70, 5, n).round(1),    # °C de la máquina
    "vibracion":   rng.normal(3, 1, n).round(2),     # mm/s
    "velocidad":   rng.normal(100, 10, n).round(0),  # piezas por hora
})
# La regla "real" (el modelo NO la conoce): más calor y vibración → más defectos
riesgo = -9 + 0.6 * (piezas.temperatura - 70) + 2.5 * (piezas.vibracion - 3)
piezas["defectuosa"] = (rng.random(n) < 1 / (1 + np.exp(-riesgo))).astype(int)

X = piezas.drop(columns="defectuosa")
y = piezas["defectuosa"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, stratify=y, random_state=0)

print(f"Piezas defectuosas: {y.mean():.2%}")
print(f"En el conjunto de prueba: {y_te.sum()} defectuosas de {len(y_te)}")

Solo **1.45 %** de las piezas sale defectuosa: los datos están muy **desbalanceados** (Lección 2.7).

### 🔮 Ejercicio 2 — El inspector flojo
Un inspector flojo decide que **todas las piezas son buenas**, sin mirarlas. La **exactitud** (*accuracy*) es el porcentaje de aciertos.

¿Qué exactitud tendrá en las 2,000 piezas de prueba? ¿Cuántas piezas defectuosas detectará?

✍️ *Mi apuesta:*

In [ ]:
flojo = np.zeros(len(y_te), dtype=int)     # 0 = "pieza buena" para todas
print(f"Exactitud del inspector flojo: {accuracy_score(y_te, flojo):.2%}")
ConfusionMatrixDisplay.from_predictions(y_te, flojo, display_labels=["buena", "defectuosa"])
plt.title("Inspector flojo"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**Exactitud = 98.55 %**… sin detectar **ni una** pieza defectuosa (las 29 se fueron al cliente).

Cómo leer la matriz: las **filas** son la realidad y las **columnas** lo que predijo el modelo. La diagonal (arriba-izquierda y abajo-derecha) son los aciertos.

```
                   predijo buena   predijo defectuosa
real buena            VN = 1971          FP = 0
real defectuosa       FN = 29            VP = 0
```

Esta es la **paradoja de la exactitud**: con clases desbalanceadas, un modelo inútil puede tener una exactitud altísima.
</details>

### 🔮 Ejercicio 3 — Un modelo de verdad
Ahora entrenamos una regresión logística (Lección 5.2) con las mediciones de la máquina.

¿Crees que le ganará **por mucho** al inspector flojo en exactitud? ¿Detectará muchas defectuosas?

✍️ *Mi apuesta:*

In [ ]:
modelo = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
pred = modelo.predict(X_te)

print(f"Exactitud del modelo: {accuracy_score(y_te, pred):.2%}")
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["buena", "defectuosa"])
plt.title("Regresión logística"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Exactitud **98.60 %** contra 98.55 %: ¡solo 0.05 puntos más! Si solo miraras la exactitud, dirías que el modelo no sirve de nada.

Pero la matriz cuenta otra historia: el modelo **detecta 10 de las 29** piezas defectuosas (VP = 10), a cambio de 9 falsas alarmas (FP = 9). Ya no es inútil… aunque se le escapan 19 (FN = 19).

**Moraleja:** la exactitud mezcla los dos tipos de error en un solo número y, cuando una clase es rara, la clase grande lo domina. Necesitamos métricas que miren la clase **que nos importa**.
</details>

---
## 3. 📖 Precisión y recall: dos preguntas distintas

Para la clase positiva (pieza defectuosa) nos hacemos dos preguntas:

| Métrica | Pregunta | Fórmula | Analogía de la alarma |
|---|---|---|---|
| **Precisión** (*precision*) | De las veces que **dije "defectuosa"**, ¿cuántas acerté? | VP / (VP + FP) | Cuando suena, ¿de verdad hay fuego? |
| **Recall** o **sensibilidad** (*recall, sensitivity*) | De todas las piezas **realmente defectuosas**, ¿cuántas encontré? | VP / (VP + FN) | De los incendios reales, ¿cuántos detectó? |

- Precisión baja → **muchas falsas alarmas** (la gente deja de hacerle caso a la alarma).
- Recall bajo → **se te escapan casos** (incendios sin alarma).

> 📝 *Recall* se traduce a veces como "exhaustividad" o "tasa de verdaderos positivos". Usaremos *recall* porque es como lo verás escrito en el código y en casi todos los textos.

### ✋ Ejercicio 4 — Calcula a mano
Con la matriz de la regresión logística (VN = 1962, FP = 9, FN = 19, VP = 10), calcula:
1. La **precisión**.
2. El **recall**.
3. Explica cada número con una frase para el jefe de planta.

✍️ *Mis cálculos:*

Luego compruébalo:

In [ ]:
print(f"Precisión: {precision_score(y_te, pred):.2f}")
print(f"Recall   : {recall_score(y_te, pred):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Precisión = 10 / (10 + 9) = **0.53**: *"cuando el sistema aparta una pieza como defectuosa, acierta un poco más de la mitad de las veces"*.
2. Recall = 10 / (10 + 19) = **0.34**: *"el sistema encuentra solo 1 de cada 3 piezas defectuosas; las otras 2 llegan al cliente"*.

Para una fábrica, ese recall es probablemente el problema principal.
</details>

### 📖 Especificidad: el recall de los negativos
Hay una tercera métrica que verás mucho en medicina:

> **Especificidad** (*specificity*) = VN / (VN + FP): de todos los **negativos reales** (piezas buenas), ¿cuántos dejé pasar correctamente?

Es el "recall de la clase negativa". En nuestro caso: 1962 / (1962 + 9) ≈ 99.5 %.

### 🔮 Ejercicio 5 — Cada métrica se puede "hacer trampa"
Comparamos tres estrategias: el inspector flojo ("todas buenas"), un inspector **paranoico** ("todas defectuosas") y la regresión logística. Antes de ejecutar, completa mentalmente: ¿quién tendrá recall de 100 %? ¿Quién tendrá especificidad de 100 %?

✍️ *Mis predicciones:*

In [ ]:
estrategias = {
    "Flojo (todas buenas)":         np.zeros(len(y_te), dtype=int),
    "Paranoico (todas defectuosas)": np.ones(len(y_te), dtype=int),
    "Regresión logística":           pred,
}
filas = []
for nombre, p in estrategias.items():
    filas.append({"estrategia": nombre,
                  "exactitud": accuracy_score(y_te, p),
                  "precisión": precision_score(y_te, p, zero_division=0),
                  "recall": recall_score(y_te, p),
                  "especificidad": recall_score(y_te, p, pos_label=0)})
pd.DataFrame(filas).set_index("estrategia").round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

- **Flojo:** especificidad 100 % (deja pasar todas las buenas), recall 0 %. Precisión 0 por convención: nunca dijo "defectuosa".
- **Paranoico:** recall **100 %** (¡las encuentra todas!), pero precisión de solo **1.5 %** y especificidad 0 %: desecharía toda la producción.
- **Logística:** el único con valores intermedios en todo.

**Moraleja:** cualquier métrica sola se puede inflar con una estrategia absurda. Por eso se reportan **en pareja**: precisión **y** recall (o recall **y** especificidad, en medicina).
</details>

---
## 4. 📖 F1: un solo número que exige equilibrio

A veces necesitas **un** número para comparar modelos. El **F1** combina precisión y recall con una **media armónica**:

> **F1** = 2 · precisión · recall / (precisión + recall)

No necesitas recordar la fórmula, solo su personalidad: **el F1 se hunde si cualquiera de los dos es bajo**. Un promedio normal se deja engañar; el F1 no.

### 🔮 Ejercicio 6 — Promedio normal vs. F1
Un modelo tiene **precisión = 1.00** (nunca da falsas alarmas) pero **recall = 0.02** (encuentra solo el 2 % de los casos).
- ¿Cuánto da el promedio normal?
- ¿Y el F1: más alto, parecido o mucho más bajo?

✍️ *Mi apuesta:*

In [ ]:
def f1(precision, recall):
    return 2 * precision * recall / (precision + recall)

for p, r in [(1.00, 0.02), (0.50, 0.50), (0.53, 0.34)]:   # ✏️ prueba tus propias parejas
    print(f"precisión={p:.2f}  recall={r:.2f}  →  promedio normal={(p + r) / 2:.2f}   F1={f1(p, r):.2f}")
print(f"\nF1 de nuestra regresión logística: {f1_score(y_te, pred):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con precisión 1.00 y recall 0.02: el promedio normal da **0.51** (¡parece un modelo regular!), pero el F1 da **0.04**: casi cero, como merece un modelo que casi nunca detecta nada.

Con 0.50 y 0.50, ambos dan 0.50: cuando hay equilibrio, coinciden. Nuestra logística tiene **F1 ≈ 0.42** (la tercera fila da 0.41 porque usa precisión y recall ya redondeados).

> ⚠️ El F1 trata la precisión y el recall como **igual de importantes**. Si un falso negativo cuesta 10 veces más que un falso positivo, el F1 no lo sabe. Elegir métrica según el costo es el tema de la Lección 4.8.
</details>

---
## 5. 📖 El tira y afloja: precisión vs. recall

La regresión logística no responde "sí/no" directamente: calcula una **probabilidad** de que la pieza sea defectuosa y, por defecto, dice "defectuosa" si esa probabilidad es **≥ 0.5**. Ese 0.5 es el **umbral**.

```
probabilidad:  0.0 ──────────────── 0.5 ──────────────── 1.0
                    "buena"          ↑ umbral   "defectuosa"
```

Si bajas el umbral, el modelo se vuelve **más desconfiado**: aparta más piezas.

### 🔮 Ejercicio 7 — Mover el umbral
Si bajamos el umbral de 0.5 a 0.1, ¿qué le pasa al **recall**? ¿Y a la **precisión**? ¿Y a la exactitud?

✍️ *Mi apuesta:*

In [ ]:
prob = modelo.predict_proba(X_te)[:, 1]   # probabilidad de "defectuosa"

for umbral in [0.5, 0.3, 0.2, 0.1, 0.05]:   # ✏️ prueba otros umbrales
    p = (prob >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_te, p).ravel()
    print(f"umbral={umbral:<5} VP={vp:>2} FP={fp:>3} FN={fn:>2}   precisión={precision_score(y_te, p):.2f}  "
          f"recall={recall_score(y_te, p):.2f}  F1={f1_score(y_te, p):.2f}  exactitud={accuracy_score(y_te, p):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Al bajar el umbral de 0.5 a 0.1:
- **Recall sube** de 0.34 a **0.79**: encuentra 23 de las 29 defectuosas.
- **Precisión baja** de 0.53 a **0.35**: las falsas alarmas pasan de 9 a 42.
- La **exactitud baja** (98.6 % → 97.6 %), ¡aunque para la fábrica este modelo probablemente es **mucho mejor**!

Es un **tira y afloja**: casi siempre que uno sube, el otro baja. El F1 es máximo cerca de 0.2 (≈ 0.51), pero el umbral "correcto" lo decide el negocio: ¿cuánto cuesta revisar una pieza buena frente a enviar una defectuosa? En la **Lección 4.3** verás todo este tira y afloja de un vistazo con las curvas ROC y Precisión-Recall, y en la **4.8** cómo elegir el umbral por costo.
</details>

---
## 6. 📖 Más de dos clases: el `classification_report`

Con varias clases, cada clase tiene **su propia** precisión, recall y F1 (tratándola como "positiva" y a las demás como "negativas"). scikit-learn los junta en un reporte.

Ejemplo: clasificar vinos en 3 cultivos según su análisis químico.

In [ ]:
X_v, y_v = load_wine(return_X_y=True)
Xv_tr, Xv_te, yv_tr, yv_te = train_test_split(X_v, y_v, test_size=0.3, stratify=y_v, random_state=0)
arbol_vino = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xv_tr, yv_tr)

print(classification_report(yv_te, arbol_vino.predict(Xv_te),
                            target_names=["cultivo_A", "cultivo_B", "cultivo_C"]))

Cómo leerlo:
- **support**: cuántos ejemplos reales hay de esa clase en la prueba.
- **macro avg**: promedio simple de las clases (todas pesan igual, aunque una sea rara).
- **weighted avg**: promedio ponderado por `support` (las clases grandes pesan más).

> 💡 Si te importan las clases raras, mira el **macro**: el ponderado puede esconder que una clase pequeña va fatal.

### ✋ Ejercicio 8 — Lee el reporte
1. ¿Qué cultivo es el que **más se le escapa** al modelo (peor recall)?
2. ¿Y en cuál se equivoca más **cuando lo dice** (peor precisión)?
3. ¿Hay relación entre esas dos respuestas?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **cultivo_B**: recall 0.71. De los 21 vinos B, el modelo reconoce solo ~15.
2. **cultivo_C**: precisión 0.74. Cuando dice "C", 1 de cada 4 veces se equivoca.
3. Sí: los vinos B que se le escapan probablemente los está llamando **C** (por eso C tiene recall alto, 0.93, pero baja precisión). Para confirmarlo mirarías la matriz de confusión de 3×3. Así se diagnostica un modelo: **qué clase confunde con cuál**.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué hacer |
|---|---|---|
| Exactitud de 98 % "¡excelente!" con una clase rara | Paradoja de la exactitud | Compara con "predecir siempre la clase mayoritaria" (Lección 4.7) y mira precisión/recall |
| Recall altísimo, nadie lo cuestiona | El modelo dice "positivo" a casi todo | Reporta siempre la **pareja** precisión + recall |
| Las métricas no cuadran con lo que esperas | Se calcularon para **la clase equivocada** | Revisa cuál es la clase positiva (`pos_label`) |
| "Mi F1 es el mejor, así que es el mejor modelo" | El F1 supone que FP y FN cuestan igual | Elige la métrica según el costo del error (Lección 4.8) |
| Promedio ponderado alto, pero una clase rara va mal | El `weighted avg` esconde las clases pequeñas | Mira el `macro avg` y el reporte por clase |
| Métricas medidas en entrenamiento | Autoengaño (Lección 4.1) | Mide siempre en datos que el modelo no vio |

El tercero es traicionero. Veámoslo con datos reales de cáncer de mama (569 tumores; Lección 5.4).

### 🔮 Ejercicio 9 — ¿Recall de qué?
Un colega entrena un árbol para detectar tumores **malignos** y te reporta: *"¡recall de 0.90!"*. Usó `recall_score(y_real, y_pred)` sin más.

En este conjunto de datos, **0 = maligno** y **1 = benigno**. scikit-learn, por defecto, calcula las métricas para la clase **1**. ¿Qué recall calculó en realidad tu colega?

✍️ *Mi respuesta:*

In [ ]:
cancer = load_breast_cancer(as_frame=True)
print("Clases:", dict(enumerate(cancer.target_names)))   # 0 = malignant, 1 = benign

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(cancer.data, cancer.target, test_size=0.3,
                                              stratify=cancer.target, random_state=0)
pred_c = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xc_tr, yc_tr).predict(Xc_te)

print(f"Lo que reportó tu colega (clase 1 = benigno) : recall = {recall_score(yc_te, pred_c):.2f}")
print(f"Lo que importa  (pos_label=0 → maligno)      : recall = {recall_score(yc_te, pred_c, pos_label=0):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Tu colega midió el recall de los tumores **benignos** (≈ 0.90). El que importa, el de los **malignos**, es ≈ 0.92. Aquí los dos números se parecen por casualidad, pero no tienen por qué: podrías ver un "recall de 0.95" que en realidad es de la clase fácil, mientras el modelo pierde muchos casos de la que importa.

**Regla:** antes de reportar cualquier métrica, pregúntate *"¿para qué clase?"*. Pasa `pos_label` explícitamente o recodifica para que **1 = la clase que quieres detectar**.
</details>

---
## 8. 🧠 Resumen en 5 puntos
1. La **matriz de confusión** separa aciertos y errores en cuatro casillas: **VP, FP, FN, VN**. Los dos errores (FP y FN) casi nunca cuestan lo mismo.
2. La **exactitud** engaña con clases desbalanceadas: "decir siempre la clase mayoritaria" puede sacar 98 % sin detectar nada.
3. **Precisión** = "cuando digo positivo, ¿acierto?"; **recall** = "de los positivos reales, ¿cuántos encuentro?". La **especificidad** es el recall de los negativos.
4. Precisión y recall **compiten**: mover el umbral sube uno y baja el otro. El **F1** los combina y se hunde si cualquiera es bajo, pero supone que ambos errores cuestan igual.
5. Decide siempre **cuál es la clase positiva** (`pos_label`) y reporta las métricas **en pareja**; con varias clases, mira el reporte por clase y el promedio **macro**.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** Un modelo de diagnóstico tiene VP = 40, FN = 10, FP = 40, VN = 910. ¿Cuál es su recall?
- a) 0.50 · b) 0.80 · c) 0.95 · d) 0.96

**2.** Con los mismos datos de la pregunta 1, ¿cuál es su precisión?
- a) 0.50 · b) 0.80 · c) 0.95 · d) 0.96

**3.** Un filtro de spam manda a la carpeta de spam correos importantes de tus clientes. ¿Qué métrica de la clase "spam" es baja?
- a) Recall · b) Precisión · c) Especificidad · d) Ninguna

**4.** Bajas el umbral de un clasificador de 0.5 a 0.2. ¿Qué es lo **más** probable?
- a) Sube la precisión y baja el recall
- b) Sube el recall y baja la precisión
- c) Suben los dos
- d) No cambia nada

**5.** En un problema con 3 clases, una de ellas con solo el 2 % de los ejemplos va fatal. ¿Qué promedio lo delata mejor?
- a) `weighted avg` · b) `macro avg` · c) La exactitud · d) Ninguno

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Recall = VP / (VP + FN) = 40 / 50 = 0.80.
2. **a)** Precisión = VP / (VP + FP) = 40 / 80 = 0.50. (Su exactitud sería 950 / 1000 = 95 %, que suena mucho mejor: ¡la paradoja otra vez!)
3. **b)** Correos buenos marcados como spam son **falsos positivos**, y los FP bajan la precisión.
4. **b)** Con un umbral más bajo el modelo dice "positivo" más veces: encuentra más casos (recall ↑) pero con más falsas alarmas (precisión ↓).
5. **b)** En el macro todas las clases pesan igual; en el ponderado, una clase del 2 % casi no influye.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Para cada sistema, ¿qué priorizarías, **precisión** o **recall**? Justifica en una frase.
1. Detector de cáncer para decidir quién se hace una biopsia.
2. Sistema que bloquea automáticamente la cuenta bancaria de un cliente.
3. Recomendador que elige 5 películas para tu pantalla de inicio.

**B. Caso de análisis 💼** Una aseguradora te presume que su modelo de "clientes que presentarán una reclamación falsa" tiene **99.3 % de exactitud**. Sabes que solo el 0.5 % de las reclamaciones son falsas. ¿Qué tres preguntas les harías antes de impresionarte?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** (Importa el razonamiento.)
1. **Recall**: un cáncer que se escapa (FN) es gravísimo; una biopsia de más (FP) es una molestia que se puede aceptar.
2. **Precisión**: bloquear la cuenta de un cliente honesto (FP) es muy dañino; mejor ser muy seguro antes de actuar.
3. **Precisión**: solo caben 5 películas, así que cada recomendación debe acertar; no importa "encontrar todas" las que te gustarían.

**B.** Por ejemplo:
1. ¿Qué exactitud tendría **"decir siempre que la reclamación es legítima"**? (99.5 %: ¡mejor que su modelo!)
2. ¿Cuál es el **recall** de la clase "falsa"? ¿Qué porcentaje de las reclamaciones falsas detecta?
3. ¿Cuál es la **precisión**? Cuando acusa a un cliente, ¿cuántas veces acierta? ¿Y se midió con datos que el modelo **no vio**?
</details>

### 9.3 🧩 Práctica integradora: "¿A quién mandamos a biopsia?"

Un hospital quiere un modelo que ayude a decidir qué tumores revisar con una biopsia. Tienes los 569 tumores del conjunto de cáncer de mama. El equipo médico tiene una prioridad clara: **que no se escape ningún tumor maligno**, aunque eso signifique algunas biopsias de más.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Pon la clase positiva donde va.** Recodificamos para que **1 = maligno** (lo que queremos detectar), así todas las métricas hablan de los malignos sin tener que acordarnos de `pos_label`.

In [ ]:
y_maligno = (cancer.target == 0).astype(int)    # 1 = maligno, 0 = benigno
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(cancer.data, y_maligno, test_size=0.3,
                                              stratify=y_maligno, random_state=0)
print(f"Malignos: {y_maligno.mean():.0%} de los tumores")
print(f"En prueba: {ym_te.sum()} malignos de {len(ym_te)}")

**Paso 2 🔮 — La línea base.** ¿Qué exactitud tiene decir siempre "benigno"? ¿Y qué recall?

✍️ *Mi apuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Exactitud ≈ **63 %** (107 benignos de 171) y recall = **0**: no detecta ningún maligno. Cualquier modelo serio debe superar esa exactitud con holgura **y** tener un recall alto.
</details>

**Paso 3 ▶️ — Dos candidatos.** Entrenamos una regresión logística (con escalado, Lección 2.3) y un árbol de profundidad 2. Ejecuta y mira la tabla.

In [ ]:
candidatos = {
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression()),
    "Árbol max_depth=2":   DecisionTreeClassifier(max_depth=2, random_state=0),
}
filas, predicciones = [], {}
for nombre, m in candidatos.items():
    p = m.fit(Xm_tr, ym_tr).predict(Xm_te)
    predicciones[nombre] = p
    vn, fp, fn, vp = confusion_matrix(ym_te, p).ravel()
    filas.append({"modelo": nombre, "exactitud": accuracy_score(ym_te, p),
                  "precisión": precision_score(ym_te, p), "recall": recall_score(ym_te, p),
                  "F1": f1_score(ym_te, p), "malignos que se escapan (FN)": fn,
                  "biopsias innecesarias (FP)": fp})
pd.DataFrame(filas).set_index("modelo").round(3)

✋ Responde:
1. ¿Qué modelo tiene mejor exactitud? ¿Y mejor recall?
2. Si solo miraras el recall, ¿podrías elegir? ¿Qué otra columna desempata?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. La **logística** gana en exactitud (≈ 95.3 % contra ≈ 86.5 %). En recall **empatan**: ambos ≈ 0.94 (60 de 64 malignos; se escapan 4).
2. No: con el mismo recall, desempata la **precisión**. El árbol manda a **19** pacientes sanas a biopsia innecesaria; la logística solo a **4**. Con igual capacidad de detección, la logística molesta mucho menos: precisión 0.94 contra 0.76.
</details>

**Paso 4 🔮 — "Que no se escape ninguno".** Se escapan 4 malignos con la logística. Bajamos su umbral para hacerla más desconfiada. Con umbral 0.1, ¿cuántos malignos crees que se escaparán? ¿Cuántas biopsias innecesarias habrá?

✍️ *Mi apuesta:*

In [ ]:
logistica = candidatos["Regresión logística"]
prob_m = logistica.predict_proba(Xm_te)[:, 1]   # probabilidad de "maligno"

for umbral in [0.5, 0.2, 0.1]:   # ✏️ prueba 0.05 o 0.01
    p = (prob_m >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(ym_te, p).ravel()
    print(f"umbral={umbral:<4}  se escapan (FN)={fn}  biopsias innecesarias (FP)={fp:>2}   "
          f"recall={recall_score(ym_te, p):.3f}  precisión={precision_score(ym_te, p):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Umbral 0.5: se escapan **4**, biopsias innecesarias **4** (recall 0.938, precisión 0.938).
- Umbral 0.2: se escapan **3**, biopsias innecesarias **13**.
- Umbral 0.1: se escapa solo **1**, a cambio de **14** biopsias innecesarias (recall 0.984, precisión 0.818).

El precio de rescatar 3 malignos más son 10 biopsias de más. Para el equipo médico, probablemente vale la pena. Pero ojo: elegimos el umbral **mirando el conjunto de prueba**, y eso es hacer trampa. En la práctica se elige con validación cruzada (Lección 4.6) y luego se confirma una sola vez con la prueba.
</details>

**Paso 5 ✋ — El informe para el comité médico.** Escribe **tres frases** para médicos que no saben ML, usando el umbral 0.1: qué detecta el modelo, cuál es su precio y qué limitación tiene. Evita las palabras "precisión" y "recall".

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

Ejemplo:
> *"De cada 64 tumores malignos, el sistema señaló 63 para biopsia; se le escapó 1. A cambio, de cada 107 tumores benignos, recomendó biopsia innecesaria en 14 (unos 13 de cada 100). Lo evaluamos con solo 171 casos y elegimos el punto de corte viéndolos, así que hay que confirmarlo con nuevos pacientes antes de usarlo, y siempre como apoyo al criterio médico, no como sustituto."*

Fíjate en que la frase usa **conteos** ("63 de 64") en lugar de porcentajes abstractos: es lo que mejor entiende un público no técnico.
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "The Confusion Matrix" y "Sensitivity and Specificity" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Classification metrics*.
- 📖 **Lectura:** Wikipedia (inglés), "Precision and recall": la imagen con el círculo de "elementos relevantes" es de las mejores explicaciones visuales.

---
⬅️ **Anterior:** [4.1 Métricas de regresión: ¿por cuánto me equivoco?](4.1-metricas-de-regresion.ipynb) · ➡️ **Siguiente:** [4.3 Umbrales, curva ROC/AUC y curva Precisión-*Recall*](4.3-umbrales-roc-y-precision-recall.ipynb)